# 🛡️ Phase 6: Independent Evaluation Environment on Google Colab

Môi trường thực thi độc lập này đóng vai trò nạp mô hình đã fine-tune từ Hugging Face Hub (`thong7d/vihsd-phobert-hate-speech`) và chạy đánh giá (Phase 6) trên tập dữ liệu kiểm tra của ViHSD.

### ✨ Đặc điểm thiết kế:
1. **Giải phóng Drive (Local-first)**: Không cần liên kết Google Drive, kết quả được tạo ngay tại bộ nhớ đệm và nén gửi về trình duyệt của bạn.
2. **Tương thích hoàn toàn (Tokenizer Sync)**: Sử dụng tách từ (`use_word_segmentation=True`) đồng bộ hoàn toàn với tokenizer của PhoBERT để đạt độ chính xác cao nhất.
3. **Tách biệt cấu hình (Modularity)**: Truyền trực tiếp repo ID thông qua CLI argument `--hf_repo_id` mà không làm thay đổi các file config huấn luyện.

### 🚀 Bước 1: Cấu hình môi trường và Tải mã nguồn

In [1]:
# Cấu hình thư mục cache cho Hugging Face
import os, sys
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf_datasets_cache"

# THÊM MỚI: Tự động nạp HF_TOKEN từ Colab Secrets để xác thực quyền truy cập repo Private
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        from huggingface_hub import login
        login(token=token, add_to_git_credential=True)
        print("🔑 Đăng nhập Hugging Face Hub thành công!")
except Exception as e:
    print(f"⚠️ Cảnh báo xác thực (Có thể repo đang để công khai): {e}")

REPO_URL  = "https://github.com/thong7d/hate-speech-detection.git"
REPO_NAME = "hate-speech-detection"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_PATH):
    print(f"📥 Đang clone repository từ: {REPO_URL}...")
    os.system(f"git clone {REPO_URL} {REPO_PATH}")
else:
    print("🔄 Cập nhật mã nguồn mới nhất từ GitHub...")
    os.system(f"cd {REPO_PATH} && git pull origin main")

print("✅ Tải mã nguồn thành công!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


🔑 Đăng nhập Hugging Face Hub thành công!
🔄 Cập nhật mã nguồn mới nhất từ GitHub...
✅ Tải mã nguồn thành công!


### 📦 Bước 2: Cài đặt thư viện dependencies

In [2]:
import os
import sys

print("📥 Đang cài đặt thư viện dependencies từ requirements.txt...")
os.system(f"pip install -q -r {REPO_PATH}/requirements.txt")

print("📥 Đang cài đặt thư viện tách từ tiếng Việt underthesea...")
os.system("pip install -q underthesea")

print("📥 Đang cài đặt dự án ở chế độ phát triển (editable mode)...")
os.system(f"pip install -q -e {REPO_PATH}")

# SỬA ĐỔI: Thêm REPO_PATH (thư mục gốc chứa gói src) vào đường dẫn hệ thống
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print("✅ Cài đặt môi trường thành công!")

📥 Đang cài đặt thư viện dependencies từ requirements.txt...
📥 Đang cài đặt thư viện tách từ tiếng Việt underthesea...
📥 Đang cài đặt dự án ở chế độ phát triển (editable mode)...
✅ Cài đặt môi trường thành công!


### 📊 Bước 3: Tải dữ liệu và Tiền xử lý tập Test (PhoBERT Tokenizer Sync)

In [3]:
from src.data.download import download_and_extract
from src.data.preprocessing import process_split

print("📥 Đang tải dữ liệu gốc từ GitHub...")
download_and_extract(f"{REPO_PATH}/configs/paths.yaml")

print("⚙️ Đang thực hiện tiền xử lý tập test (use_word_segmentation=False)...")
process_split(
    input_path=f"{REPO_PATH}/data/raw/vihsd/test.csv",
    output_path=f"{REPO_PATH}/data/processed/test.parquet",
    use_word_segmentation=False
)

print("✅ Tiền xử lý dữ liệu kiểm tra thành công! Tập test đã lưu dưới dạng Parquet.")

📥 Đang tải dữ liệu gốc từ GitHub...
Download successful.
Extracting files...
[OK] Extracted: test.csv -> /content/hate-speech-detection/data/raw/vihsd/test.csv
[OK] Extracted: dev.csv -> /content/hate-speech-detection/data/raw/vihsd/dev.csv
[OK] Extracted: train.csv -> /content/hate-speech-detection/data/raw/vihsd/train.csv
[SUCCESS] Phase 1: Data Acquisition completed successfully.
⚙️ Đang thực hiện tiền xử lý tập test (giữ nguyên use_word_segmentation=True)...
✅ Tiền xử lý dữ liệu kiểm tra thành công! Tập test đã lưu dưới dạng Parquet.


### 🧠 Bước 4: Thực hiện đánh giá mô hình qua CLI

In [4]:
import os, shutil
# Tối ưu hóa phi trạng thái: Dọn dẹp kết quả và file zip cũ trước khi đánh giá mới
results_dir = f"{REPO_PATH}/results"
if os.path.exists(results_dir):
    shutil.rmtree(results_dir)
os.makedirs(results_dir, exist_ok=True)

zip_file = "/content/evaluation_results.zip"
if os.path.exists(zip_file):
    os.remove(zip_file)

print("🚀 Bắt đầu quá trình đánh giá mô hình thong7d/vihsd-xlmr-hate-speech...")
# Đánh giá mô hình nạp từ HF Hub và ghi đè tách từ là False
!cd /content/hate-speech-detection && python -m src.evaluation.evaluate --config configs/train.yaml --hf_repo_id thong7d/vihsd-xlmr-hate-speech --use_word_segmentation False

🚀 Bắt đầu quá trình đánh giá mô hình thong7d/vihsd-phobert-hate-speech...
config.json: 100% 884/884 [00:00<00:00, 2.83MB/s]
tokenizer_config.json: 100% 1.17k/1.17k [00:00<00:00, 500kB/s]
added_tokens.json: 100% 22.0/22.0 [00:00<00:00, 85.5kB/s]
special_tokens_map.json: 100% 167/167 [00:00<00:00, 713kB/s]
vocab.txt: 100% 895k/895k [00:00<00:00, 6.39MB/s]
bpe.codes: 100% 1.14M/1.14M [00:00<00:00, 11.2MB/s]
model/model.safetensors: 100% 540M/540M [00:32<00:00, 16.8MB/s]
Loading weights: 100% 201/201 [00:00<00:00, 1096.82it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
[EVAL] Error analysis written: 988 errors -> /content/hate-speech-detection/results/error_analysis.csv
{'accuracy': 0.8511, 'macro_f1': 0.6336, 'weighted_f1': 0.8526, 'precision_CLEAN': 0.9269, 'recall_CLEAN': 0.9213, 'f1_CLEAN': 0.9241, 'precision_OFFENSIVE': 0.3891, 'recall_OFFENSIVE': 0.4347, 'f1_OFFENSIVE': 0.4106, 'precision_HATE': 0.574, 'recall_HATE': 0.5581, 'f1_HATE': 0.566, 'critical_f1': 0.4

In [5]:
from huggingface_hub import list_repo_files
try:
    files = list_repo_files(repo_id="thong7d/vihsd-phobert-hate-speech")
    print("📂 Các file hiện có trên HF Hub:")
    for f in files:
        print(f" - {f}")
except Exception as e:
    print(f"❌ Lỗi quét repo: {e}")

📂 Các file hiện có trên HF Hub:
 - .gitattributes
 - label_mapping.json
 - metadata.json
 - metrics.json
 - model/added_tokens.json
 - model/bpe.codes
 - model/config.json
 - model/model.safetensors
 - model/special_tokens_map.json
 - model/tokenizer_config.json
 - model/training_args.bin
 - model/vocab.txt
 - model_card.md


### 📥 Bước 5: Nén và Tải kết quả về máy cục bộ

In [6]:
import shutil
import os
from google.colab import files

# Schema guard: Đảm bảo thư mục tồn tại trên ổ đĩa trước khi ra lệnh nén
os.makedirs(f"{REPO_PATH}/results", exist_ok=True)

print("📦 Đang nén thư mục kết quả đánh giá (results/)...")
shutil.make_archive("/content/evaluation_results", "zip", f"{REPO_PATH}/results")

print("📥 Đang tải file evaluation_results.zip về máy cục bộ của bạn...")
files.download("/content/evaluation_results.zip")
print("✅ Đã kích hoạt hộp thoại tải xuống thành công!")

📦 Đang nén thư mục kết quả đánh giá (results/)...
📥 Đang tải file evaluation_results.zip về máy cục bộ của bạn...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Đã kích hoạt hộp thoại tải xuống thành công!


### 📝 Bước 6: Hiển thị báo cáo đánh giá trực tiếp

In [7]:
from IPython.display import Markdown, display

report_path = f"{REPO_PATH}/results/evaluation_report.md"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        content = f.read()
    print("\n" + "="*40 + " BÁO CÁO ĐÁNH GIÁ CHI TIẾT " + "="*40 + "\n")
    display(Markdown(content))
else:
    print("⚠️ Không tìm thấy tệp báo cáo evaluation_report.md!")


======================================== BÁO CÁO ĐÁNH GIÁ CHI TIẾT ========================================



# Evaluation Report

- Accuracy: 0.8511
- Macro F1: 0.6336
- Weighted F1: 0.8526
- Critical F1: 0.4883
- Offensive Priority F1: 0.4572
- Balanced Critical F1: 0.5174
- AUC-ROC Macro: None

## Per-Class Metrics

| Class | Precision | Recall | F1 | AUC-ROC |
|---|---:|---:|---:|---:|
| CLEAN | 0.9269 | 0.9213 | 0.9241 | N/A |
| OFFENSIVE | 0.3891 | 0.4347 | 0.4106 | N/A |
| HATE | 0.5740 | 0.5581 | 0.5660 | N/A |

## Confusion Matrix

```text
[[5072, 221, 212], [178, 193, 73], [222, 82, 384]]
```

## Error Analysis Summary

- Total errors: 988 / 6637
- Error rate: 0.1489
- False positive (CLEAN → toxic): 433
- False negative (OFFENSIVE → CLEAN): 178
- False negative (HATE → CLEAN): 222
- OFFENSIVE ↔ HATE confusion: 155

## Error Counts

| Expected | Missed as CLEAN | Predicted as toxic from CLEAN |
|---|---:|---:|
| CLEAN | 0 | 0 |
| OFFENSIVE | 178 | 221 |
| HATE | 222 | 212 |
